# Project: Compliance Document Reviewer
**Brief from Paras:**
Watches Drive folder for new contracts. Reads, checks against Notion compliance checklist,
flags missing clauses, drafts summary email, creates calendar reminder.
MANDATORY HITL before email send.

This is a SCAFFOLD. The supervisor + graph wiring is complete; worker logic for some nodes is marked TODO.
Use Project #2 (`02_support_resolver/support_resolver.ipynb`) as your reference implementation.

## Submission checklist
- [ ] All worker TODOs filled in
- [ ] Composio actions verified against docs.composio.dev
- [ ] HITL where destructive actions occur
- [ ] Checkpoint persistence configured
- [ ] Graph diagram saved as graph.png
- [ ] README.md with architecture + example run


## 1. Setup

In [ ]:
import os, sqlite3
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(".env")

assert os.getenv("OPENAI_API_KEY")
assert os.getenv("COMPOSIO_API_KEY"), "Sign up at composio.dev and connect required apps in their dashboard"
print("env OK")


## 2. State schema

In [ ]:
from typing import TypedDict, Annotated, Optional, Literal
from langgraph.graph.message import add_messages
from langchain_core.messages import AnyMessage, HumanMessage, AIMessage, SystemMessage

class ComplianceReviewerState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
    next_worker: str
    drive_file_id: str
    doc_text: Optional[str]
    flagged_clauses: list[str]
    draft_email: Optional[str]
    reminder_event_id: Optional[str]


## 3. LLM and Composio toolset

In [ ]:
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent
from composio_langgraph import Action, ComposioToolSet

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
toolset = ComposioToolSet()


## 4. Workers

In [ ]:
# doc_reader - Drive document reader
doc_reader_tools = toolset.get_tools(actions=[Action.GOOGLEDRIVE_FIND_FILE])
doc_reader_agent = create_react_agent(llm, doc_reader_tools, prompt='''Fetch the file content from Drive. Return doc_text.''')
def doc_reader_node(state):
    result = doc_reader_agent.invoke({'messages': state['messages']})
    last = result['messages'][-1]
    return {'messages': [AIMessage(content=last.content, name='doc_reader')]}


# checklist_matcher - Notion checklist matcher
checklist_matcher_tools = toolset.get_tools(actions=[Action.NOTION_QUERY_DATABASE])
checklist_matcher_agent = create_react_agent(llm, checklist_matcher_tools, prompt='''Pull compliance checklist from Notion. Compare to doc_text. Return list of missing or risky clauses.''')
def checklist_matcher_node(state):
    result = checklist_matcher_agent.invoke({'messages': state['messages']})
    last = result['messages'][-1]
    return {'messages': [AIMessage(content=last.content, name='checklist_matcher')]}


# email_drafter - Email drafter (pure LLM)
# No Composio actions; pure LLM or custom logic.
# TODO: implement email_drafter_node(state) following the pattern in 02_support_resolver.
def email_drafter_node(state):
    """Draft a summary email to the reviewer covering each flagged clause with severity."""
    raise NotImplementedError('TODO: implement email_drafter_node')


# email_sender - Gmail sender (HITL gated)
email_sender_tools = toolset.get_tools(actions=[Action.GMAIL_SEND_EMAIL])
email_sender_agent = create_react_agent(llm, email_sender_tools, prompt='''Send the approved email. Compile graph with interrupt_before=['email_sender'].''')
def email_sender_node(state):
    result = email_sender_agent.invoke({'messages': state['messages']})
    last = result['messages'][-1]
    return {'messages': [AIMessage(content=last.content, name='email_sender')]}


# reminder_creator - Calendar reminder
reminder_creator_tools = toolset.get_tools(actions=[Action.GOOGLECALENDAR_CREATE_EVENT])
reminder_creator_agent = create_react_agent(llm, reminder_creator_tools, prompt='''Create a 7-day follow-up reminder.''')
def reminder_creator_node(state):
    result = reminder_creator_agent.invoke({'messages': state['messages']})
    last = result['messages'][-1]
    return {'messages': [AIMessage(content=last.content, name='reminder_creator')]}


## 5. Supervisor + router

In [ ]:
def supervisor(state) -> dict:
    if state.get('doc_text') is None: return {'next_worker': 'doc_reader'}
    if not state['flagged_clauses'] and 'checklist_done' not in str(state['messages']):
        return {'next_worker': 'checklist_matcher'}
    if state.get('draft_email') is None: return {'next_worker': 'email_drafter'}
    if 'email sent' not in (state['messages'][-1].content.lower() if state['messages'] else ''):
        return {'next_worker': 'email_sender'}  # HITL pause happens here
    if state.get('reminder_event_id') is None: return {'next_worker': 'reminder_creator'}
    return {'next_worker': 'DONE'}

def route(state) -> str:
    nxt = state['next_worker']
    if nxt in {'checklist_matcher', 'doc_reader', 'email_sender', 'reminder_creator', 'email_drafter'}:
        return nxt
    return '__end__'


## 6. Build graph + checkpoint persistence

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.sqlite import SqliteSaver

g = StateGraph(globals()[[k for k in globals() if k.endswith('State') and k != 'AgentState'][0]])
g.add_node("supervisor", supervisor)
g.add_node("doc_reader", doc_reader_node)
g.add_node("checklist_matcher", checklist_matcher_node)
g.add_node("email_drafter", email_drafter_node)
g.add_node("email_sender", email_sender_node)
g.add_node("reminder_creator", reminder_creator_node)

g.add_edge(START, "supervisor")
g.add_conditional_edges("supervisor", route, {
    "doc_reader": "doc_reader",
    "checklist_matcher": "checklist_matcher",
    "email_drafter": "email_drafter",
    "email_sender": "email_sender",
    "reminder_creator": "reminder_creator",
    "__end__": END,
})
g.add_edge("doc_reader", "supervisor")
g.add_edge("checklist_matcher", "supervisor")
g.add_edge("email_drafter", "supervisor")
g.add_edge("email_sender", "supervisor")
g.add_edge("reminder_creator", "supervisor")

conn = sqlite3.connect("09_compliance_reviewer.db", check_same_thread=False)
app = g.compile(checkpointer=SqliteSaver(conn))


## 7. Visualise (submission requirement)

In [ ]:
from IPython.display import Image, display
try:
    png = app.get_graph().draw_mermaid_png()
    Path("graph.png").write_bytes(png)
    display(Image("graph.png"))
except Exception as e:
    print("ASCII fallback:")
    print(app.get_graph().draw_ascii())


## 8. End-to-end demo

In [ ]:
config = {'configurable': {'thread_id': '09_compliance_reviewer-demo-1'}, 'recursion_limit': 30}

result = app.invoke(
    {'drive_file_id': '1AbC...',
        'messages': [HumanMessage(content='New contract uploaded to /compliance/inbox')]},
    config=config,
)

print("=== FINAL STATE ===")
for k, v in result.items():
    if k != 'messages':
        print(f"{k}: {str(v)[:150]}")
print("\n=== MESSAGE TRACE ===")
for m in result['messages']:
    label = getattr(m, 'name', m.type)
    print(f"[{label}] {m.content[:200]}")
